# Batch optotagging metric generation

Generate the per-probe `*_laser_response_metrics.csv` files for a **list of
sessions**. This is the batch equivalent of Steps 1–4 of
`optotagging_Anna_nwb.ipynb`: for each session it loads the ephys NWB, locates the
raw `ecephys_clipped` asset (NIDAQ onsets + `*opto.csv`), computes laser-response
metrics for every QC unit on each stimulated probe, and writes one CSV per probe
to `SAVE_FOLDER`.

Once the CSVs exist, use `optotagging_Anna_nwb.ipynb` (Step 6, *Option A*) to append
the results back onto each NWB and select tagged units. Failures are caught
per session so one bad session does not stop the batch.

## Setup

In [1]:
import sys
from pathlib import Path

%load_ext autoreload
%autoreload 2

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

import numpy as np
import pandas as pd

from nwb_utils import NWBUtils
from optotagging_Anna_nwb import (
    OptotaggingAnalysisNWB,
    find_recording_clipped_folder,
)
import optotagging_Anna_nwb_plotting as opto_plot

print(f"✅ Modules loaded from: {MODULE_PATH}")


✅ Modules loaded from: /root/capsule/src/aind_dft_ephys_analysis


## Configuration

List the sessions to process and set the NIDAQ / output parameters (same meaning
as in the single-session notebook).

In [2]:
SESSION_NAMES = [
    "ecephys_839480_2026-06-03_15-09-14_sorted-bandpass_2026-07-18_00-07-17",
    "ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58",
    "ecephys_839480_2026-06-05_14-36-28_sorted-bandpass_2026-07-18_00-37-38",
    "ecephys_839483_2026-05-28_14-47-06_sorted-bandpass_2026-07-17_20-18-46"
    # add more sorted session names here...
]

SAVE_FOLDER = "/root/capsule/scratch/opto_tagging_Anna"

LASER_EVENT_ID = "2"   # NIDAQ channel-2 digital-input label
OPTO_RECORDING = 0     # segment index with the laser stimulation
FLIP_NIDAQ = False     # subtract 0.5 s if the sync signal was flipped
PRE_OPTO_DURATION = None  # set to a float (s) to compute pre-stim ISI / rate

# ---- Plotting (tagged-unit raster / pulse plots, same as the single-session nb) ----
MAKE_PLOTS = True          # also save tagged-unit raster / pulse plots per session
RED_MIN_SIG_PULSES = 4     # red: >= 4 significant pulses
BLUE_MIN_SIG_PULSES = 5    # blue: == 5 significant pulses
MAX_JITTER = 0.01          # s
MAX_ISI = 0.5              # pre-stim ISI-violation ratio

print(f"{len(SESSION_NAMES)} session(s) to process -> {SAVE_FOLDER}")


4 session(s) to process -> /root/capsule/scratch/opto_tagging_Anna


## Per-session metric generation

The helper loads one session, computes metrics for every stimulated probe, saves
the CSVs, and (when `MAKE_PLOTS` is set) also saves the tagged-unit raster / pulse
plots for each probe — the same figures produced by the single-session notebook.
It always closes the NWB IO handle.


In [3]:
def tagged_units(metrics, trial_type, min_sig_pulses=4, max_jitter=0.006, max_isi=0.5):
    """Select tagged units matching Anna's main.py criteria (per trial type)."""
    q = []
    if f"{trial_type}_train_max_num_sig_pulses" in metrics.columns:
        q.append(f"{trial_type}_train_max_num_sig_pulses >= {min_sig_pulses}")
    if f"{trial_type}_train_best_mean_jitter" in metrics.columns:
        q.append(f"{trial_type}_train_best_mean_jitter < {max_jitter}")
    if "pre_stim_isi_ratio" in metrics.columns:
        q.append(f"pre_stim_isi_ratio < {max_isi}")
    if not q:
        return metrics.iloc[0:0]
    return metrics.query(" and ".join(q))


def plot_tagged_units_for_probe(
    analysis, metrics, trial_types, probe, save_folder,
    red_min_sig_pulses=4, blue_min_sig_pulses=5, max_jitter=0.01, max_isi=0.5,
):
    """Save raster + pulse plots for tagged units on one probe (mirrors Step 5).

    Returns the number of tagged units plotted.
    """
    red_types = [t for t in trial_types if "red" in t]
    blue_types = [t for t in trial_types if "blue" in t]
    n_plotted = 0
    red_indices = set()

    # Red types first
    for trial_type in red_types:
        tagged = tagged_units(
            metrics, trial_type, min_sig_pulses=red_min_sig_pulses,
            max_jitter=max_jitter, max_isi=max_isi,
        )
        red_indices.update(tagged.index.tolist())
        unit_ids = tagged["unit_id"].astype(int).tolist()
        print(f"    {probe} / {trial_type}: {len(unit_ids)} tagged units -> {unit_ids}")
        if not unit_ids:
            continue
        base = f"{analysis.session}_{probe}_{trial_type}_responsive"
        opto_plot.multi_unit_raster_plot(
            analysis, unit_ids, trial_types, probe, base, save_folder=save_folder
        )
        opto_plot.multi_unit_pulse_plot(
            analysis, unit_ids, metrics, trial_types, probe, base + "_pulse_plot",
            save_folder=save_folder,
        )
        n_plotted += len(unit_ids)

    # Blue types (require all pulses significant; exclude red-responsive units)
    for trial_type in blue_types:
        tagged = tagged_units(
            metrics, trial_type, min_sig_pulses=blue_min_sig_pulses,
            max_jitter=max_jitter, max_isi=max_isi,
        )
        tagged = tagged[~tagged.index.isin(red_indices)]
        unit_ids = tagged["unit_id"].astype(int).tolist()
        print(f"    {probe} / {trial_type}: {len(unit_ids)} tagged units -> {unit_ids}")
        if not unit_ids:
            continue
        base = f"{analysis.session}_{probe}_{trial_type}_responsive"
        opto_plot.multi_unit_raster_plot(
            analysis, unit_ids, trial_types, probe, base, save_folder=save_folder
        )
        opto_plot.multi_unit_pulse_plot(
            analysis, unit_ids, metrics, trial_types, probe, base + "_pulse_plot",
            save_folder=save_folder,
        )
        n_plotted += len(unit_ids)

    return n_plotted


def generate_metrics_for_session(
    session_name,
    save_folder,
    laser_event_id="2",
    opto_recording=0,
    flip_nidaq=False,
    pre_opto_duration=None,
    make_plots=True,
    red_min_sig_pulses=4,
    blue_min_sig_pulses=5,
    max_jitter=0.01,
    max_isi=0.5,
):
    """Compute and save per-probe laser-response metric CSVs for one session.

    When ``make_plots`` is True, also saves tagged-unit raster / pulse plots for
    every stimulated probe (same figures as the single-session notebook).
    """
    nwb_data = NWBUtils.read_ephys_nwb(session_name=session_name)
    if nwb_data is None:
        raise RuntimeError(f"Failed to load ephys NWB for '{session_name}'.")

    try:
        clipped = find_recording_clipped_folder(session_name)
        analysis = OptotaggingAnalysisNWB(
            nwb_data=nwb_data,
            session_name=session_name,
            recording_clipped_folder=clipped,
            laser_event_id=laser_event_id,
            opto_recording=opto_recording,
            flip_NIDAQ=flip_nidaq,
        )

        if "type" in analysis.trial_ids.columns:
            trial_types = list(np.unique(analysis.trial_ids["type"]))
        else:
            trial_types = ["all"]
        powers = (
            list(np.unique(analysis.trial_ids["power"]))
            if "power" in analysis.trial_ids.columns
            else [None]
        )
        trials_query = {"type": trial_types, "power": powers}
        suffixes = [None, "mW"]

        Path(save_folder).mkdir(parents=True, exist_ok=True)
        saved = []
        n_plotted = 0
        for probe in analysis.get_stream_names():
            metrics = analysis.one_probe_laser_responses(
                trials_query=trials_query,
                probe=probe,
                suffixes=suffixes,
                ignore_onset_offset=True,
                pre_opto_duration=pre_opto_duration,
            )
            if len(metrics) == 0:
                continue
            metrics = OptotaggingAnalysisNWB.add_best_power_columns(metrics, trial_types)
            out_csv = Path(save_folder) / f"{analysis.session}_{probe}_laser_response_metrics.csv"
            metrics.to_csv(out_csv, index=False)
            saved.append(str(out_csv))
            print(f"    saved {out_csv.name} ({len(metrics)} units)")

            if make_plots:
                n_plotted += plot_tagged_units_for_probe(
                    analysis, metrics, trial_types, probe, save_folder,
                    red_min_sig_pulses=red_min_sig_pulses,
                    blue_min_sig_pulses=blue_min_sig_pulses,
                    max_jitter=max_jitter, max_isi=max_isi,
                )

        return {
            "session": session_name,
            "n_qc_units": int(len(analysis.qc_units)),
            "n_onsets": int(len(analysis.laser_onset_times)),
            "trial_types": trial_types,
            "n_csvs": len(saved),
            "n_tagged_plotted": n_plotted,
        }
    finally:
        if hasattr(nwb_data, "io"):
            nwb_data.io.close()


## Run the batch

In [4]:
summary = []
for session_name in SESSION_NAMES:
    print(f"\n=== {session_name} ===")
    try:
        info = generate_metrics_for_session(
            session_name,
            SAVE_FOLDER,
            laser_event_id=LASER_EVENT_ID,
            opto_recording=OPTO_RECORDING,
            flip_nidaq=FLIP_NIDAQ,
            pre_opto_duration=PRE_OPTO_DURATION,
            make_plots=MAKE_PLOTS,
            red_min_sig_pulses=RED_MIN_SIG_PULSES,
            blue_min_sig_pulses=BLUE_MIN_SIG_PULSES,
            max_jitter=MAX_JITTER,
            max_isi=MAX_ISI,
        )
        info["status"] = "ok"
        print(
            f"  QC units: {info['n_qc_units']}, onsets: {info['n_onsets']}, "
            f"CSVs written: {info['n_csvs']}, tagged units plotted: {info['n_tagged_plotted']}"
        )
    except Exception as exc:  # noqa: BLE001 - keep the batch going
        info = {"session": session_name, "status": f"ERROR: {exc}"}
        print(f"  ERROR: {exc}")
    summary.append(info)

summary_df = pd.DataFrame(summary)
summary_df



=== ecephys_839480_2026-06-03_15-09-14_sorted-bandpass_2026-07-18_00-07-17 ===
Found ephys NWB: /root/capsule/data/ecephys_839480_2026-06-03_15-09-14_sorted-bandpass_2026-07-18_00-07-17/nwb/ecephys_839480_2026-06-03_15-09-14_experiment1_recording1.nwb
Successfully read ephys NWB from: /root/capsule/data/ecephys_839480_2026-06-03_15-09-14_sorted-bandpass_2026-07-18_00-07-17/nwb/ecephys_839480_2026-06-03_15-09-14_experiment1_recording1.nwb
default_qc flag passed 0 units; reconstructed QC from raw metrics (presence_ratio >= 0.8, isi_violations_ratio <= 0.5).
Number of units passing QC: 239
    saved ecephys_839480_2026-06-03_15-09-14_sorted-bandpass_2026-07-18_00-07-17_Probe A_laser_response_metrics.csv (67 units)
    Probe A / external_red: 2 tagged units -> [387, 448]


/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb_plotting.py:205: UserWarning: constrained_layout not applied because axes sizes collapsed to zero.  Try making figure larger or Axes decorations smaller.
  fig.savefig(out, dpi=150)


/root/capsule/scratch/opto_tagging_Anna/ecephys_839480_2026-06-03_15-09-14_sorted-bandpass_2026-07-18_00-07-17_Probe A_external_red_responsive.png saved


/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb_plotting.py:297: UserWarning: constrained_layout not applied because axes sizes collapsed to zero.  Try making figure larger or Axes decorations smaller.
  fig.savefig(out, dpi=150)


/root/capsule/scratch/opto_tagging_Anna/ecephys_839480_2026-06-03_15-09-14_sorted-bandpass_2026-07-18_00-07-17_Probe A_external_red_responsive_pulse_plot.png saved
    Probe A / external_blue: 51 tagged units -> [104, 108, 114, 118, 123, 124, 137, 138, 141, 145, 175, 177, 183, 196, 198, 199, 200, 357, 362, 364, 365, 371, 372, 376, 386, 393, 426, 428, 585, 590, 598, 603, 620, 622, 627, 640, 652, 657, 678, 680, 795, 805, 806, 820, 828, 839, 840, 851, 859, 863, 881]


...

/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb_plotting.py:205: UserWarning: constrained_layout not applied because axes sizes collapsed to zero.  Try making figure larger or Axes decorations smaller.
  fig.savefig(out, dpi=150)


/root/capsule/scratch/opto_tagging_Anna/ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58_Probe A_external_blue_responsive.png saved


/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb_plotting.py:297: UserWarning: constrained_layout not applied because axes sizes collapsed to zero.  Try making figure larger or Axes decorations smaller.
  fig.savefig(out, dpi=150)


/root/capsule/scratch/opto_tagging_Anna/ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58_Probe A_external_blue_responsive_pulse_plot.png saved
    saved ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58_Probe B_laser_response_metrics.csv (240 units)
    Probe B / external_red: 0 tagged units -> []
    Probe B / external_blue: 0 tagged units -> []
  QC units: 385, onsets: 420, CSVs written: 2, tagged units plotted: 91

=== ecephys_839480_2026-06-05_14-36-28_sorted-bandpass_2026-07-18_00-37-38 ===
Found ephys NWB: /root/capsule/data/ecephys_839480_2026-06-05_14-36-28_sorted-bandpass_2026-07-18_00-37-38/nwb/ecephys_839480_2026-06-05_14-36-28_experiment1_recording1.nwb
Successfully read ephys NWB from: /root/capsule/data/ecephys_839480_2026-06-05_14-36-28_sorted-bandpass_2026-07-18_00-37-38/nwb/ecephys_839480_2026-06-05_14-36-28_experiment1_recording1.nwb
default_qc flag passed 0 units; reconstructed QC from raw metrics (presence_ratio >= 0.8, isi

/opt/conda/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/opt/conda/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/opt/conda/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/opt/conda/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/opt/conda/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/opt/conda/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/opt/conda/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:172: Runtim

## Summary

`summary_df` lists each session's status, QC-unit count, laser-onset count and the
number of CSVs written. Rows with a non-`ok` status hit an error (missing raw
asset, NIDAQ mismatch, etc.) and were skipped.

In [ ]:
ok = summary_df[summary_df["status"] == "ok"] if len(summary_df) else summary_df
failed = summary_df[summary_df["status"] != "ok"] if len(summary_df) else summary_df
print(f"Succeeded: {len(ok)} / {len(summary_df)} sessions")
if len(failed):
    print("Failed sessions:")
    for _, r in failed.iterrows():
        print(f"  {r['session']}: {r['status']}")